In [4]:

import sys
import os
import importlib

# Remove all lingo_to_pyomo modules from cache to force reload
modules_to_remove = [m for m in sys.modules if 'lingo_to_pyomo' in m or 'json_parser' in m or 'lingo' in m or 'notebook_generator' in m or 'excel_parser' in m]
for m in modules_to_remove:
    del sys.modules[m]

print(f"Cleared {len(modules_to_remove)} cached modules")

# Now re-import
src_path = os.path.abspath("../src")
sys.path.insert(0, src_path)

from lingo_parser.parser import *
from lingo_parser.transformer import *
from pyomo_generator.json_parser import *
from notebook_generator.notebook_construct import *
from excel_parser.excel_module import *

print("Reloaded all modules")


Cleared 8 cached modules
Reloaded all modules


In [6]:
import sys
import os

src_path = os.path.abspath("../src")
sys.path.append(src_path)

from lingo_parser.parser import *
from lingo_parser.transformer import *


from pyomo_generator.json_parser import *
from notebook_generator.notebook_construct import *

from excel_parser.excel_module import *

In [22]:
tree = parse_lingo_model("../data/Cardoza.lng")
model_dict = LingoModelTransformer2().transform(tree)
pyomo_code = generate_pyomo_code(model_dict)
print(pyomo_code)

from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.MACHINES = Set(initialize=[1, 2])
model.PRODUITS = Set(initialize=['remorcage', 'stabilisateur'])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.MACHINES for j in model.PRODUITS])

#==============================================================================
# PARAMETERS
#==============================================================================

model.disponibilite = Param(model.MACHINES, initialize={1: 16.0, 2: 15.0}, within=NonNegativeReals)
model.gain = Param(model.PRODUITS, initialize={'remorcage': 130.0, 'stabilisateur': 150.0}, within=NonNegativeReals)
model.temps = Param(model.MACHINES, model.PRODUITS, initialize={(1, 'remorcage'): 3.2, (1, 'stabilisateur'): 2.4, (2, 'remorcage'): 2.0, (2, 'stabilisateur'): 3.0}, within=NonNegativeReals

In [28]:
from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.MACHINES = Set(initialize=[1, 2])
model.PRODUITS = Set(initialize=['remorcage', 'stabilisateur'])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.MACHINES for j in model.PRODUITS])

#==============================================================================
# PARAMETERS
#==============================================================================

model.disponibilite = Param(model.MACHINES, initialize={1: 16.0, 2: 15.0}, within=NonNegativeReals)
model.gain = Param(model.PRODUITS, initialize={'remorcage': 130.0, 'stabilisateur': 150.0}, within=NonNegativeReals)
model.temps = Param(model.MACHINES, model.PRODUITS, initialize={(1, 'remorcage'): 3.2, (1, 'stabilisateur'): 2.4, (2, 'remorcage'): 2.0, (2, 'stabilisateur'): 3.0}, within=NonNegativeReals)

#==============================================================================
# VARIABLES
#==============================================================================

model.x = Var(model.PRODUITS, domain=NonNegativeReals)

#==============================================================================
# CONSTRAINTS
#==============================================================================

model.c_for_0 = ConstraintList()
for m in model.MACHINES:
    model.c_for_0.add(sum(model.temps[m,p] * model.x[p] for p in model.PRODUITS) <= model.disponibilite[m])

#==============================================================================
# OBJECTIVE
#==============================================================================

model.obj = Objective(expr=sum(model.gain[p] * model.x[p] for p in model.PRODUITS), sense=maximize)

In [29]:


# === Résolution ===
solver = SolverFactory('highs')  # ou 'cbc', 'gurobi', 'cplex' selon ton install
solver.solve(model, tee=True)


print("Valeur objectif : ",value(model.obj))
for v in model.component_objects(Var, active=True):
    print(f'Variable set: {v}')
    for index in v:
        print(f'   {index} = {v[index].value}')

Valeur objectif :  825.0000000000001
Variable set: x
   remorcage = 2.4999999999999996
   stabilisateur = 3.3333333333333344


In [5]:


def generate_notebook() :

    file_test = "notetest"
    tree = parse_lingo_model("../data/Cardoza.lng")
    model_dict = LingoModelTransformer2().transform(tree)

    pyomo_code = generate_pyomo_code(model_dict)

    generate_pyomo_notebook(pyomo_code, solver="highs", filename=f"{file_test}.ipynb")

generate_notebook()

✅ Notebook généré : notetest.ipynb


In [45]:
convert_lingo_ole_to_explicit("../data/Regime.lng")

'/Users/joaquim/Documents/CODE/Lingpy/lingo_to_pyomo/data/Regime_explicit.lng'

In [63]:


tree = parse_lingo_model("../data/Regime_explicit_clean.lng")
model_dict = LingoModelTransformer2().transform(tree)

save_pyomo_data_to_json(model_dict)



pyomo_code = generate_pyomo_code(model_dict, external_data=True)

#print(pyomo_code)
local_vars = {}
exec(pyomo_code, local_vars)
model = local_vars["model"]

# Résout le modèle avec un solveur (par défaut glpk ou cbc)
solver = SolverFactory("highs")  # ou 'cbc' si glpk n'est pas disponible
result = solver.solve(model, tee=False)


print(model.obj())

90.0


In [64]:
print(pyomo_code)

from pyomo_generator.json_parser import load_pyomo_data
data = load_pyomo_data('./data/pyomo_data.json')

from pyomo.environ import *

model = ConcreteModel()


#==============================================================================
# SETS
#==============================================================================

model.ALIMENTS = Set(initialize=data['sets']['ALIMENTS'])
model.INGREDIENTS = Set(initialize=data['sets']['INGREDIENTS'])
model.ARCS = Set(dimen=2, initialize=[(i,j) for i in model.ALIMENTS for j in model.INGREDIENTS])

#==============================================================================
# PARAMETERS
#==============================================================================

model.Prix = Param(model.ALIMENTS, initialize=data['params']['Prix'], within=NonNegativeReals)
model.Calories = Param(model.ALIMENTS, initialize=data['params']['Calories'], within=NonNegativeReals)
model.DIETEJOUR = Param(model.INGREDIENTS, initialize=data['params']['DIETEJOUR

In [33]:


# === Résolution ===
solver = SolverFactory('highs')  # ou 'cbc', 'gurobi', 'cplex' selon ton install
solver.solve(model, tee=True)


print("Valeur objectif : ",value(model.obj))
for v in model.component_objects(Var, active=True):
    print(f'Variable set: {v}')
    for index in v:
        print(f'   {index} = {v[index].value}')

ValueError: Unexpected expression (type Numeric_GetItemExpression)